In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from typing import Optional

def plot_pert_orig_diff_side_by_side(
    pert_y: torch.Tensor,          # shape: (N, H, W, T)
    orig_y: torch.Tensor,          # shape: (N, H, W, T)
    sample_idx: int = 0,
    rows: int = 5,
    cols: int = 5,                 # 每行显示 cols 组，每组3张图 (pert, orig, diff)
    cmap_pert: str = "viridis",
    cmap_orig: str = "viridis",
    cmap_diff: str = "coolwarm",
    origin: str = "lower",
    data_name: str = "",
    shared_scale: bool = True      # 是否使用统一的 color scale
) -> plt.Figure:
    """
    每行显示 cols 组，每组 (pert, orig, diff) 三张图。
    shared_scale=True 时，三图共用一个 (vmin, vmax)；
    shared_scale=False 时，每张图自己用自己的色标。
    """
    if pert_y.ndim != 4 or orig_y.ndim != 4:
        raise ValueError(f"expect (N,H,W,T), got {tuple(pert_y.shape)} and {tuple(orig_y.shape)}")
    if sample_idx < 0 or sample_idx >= min(pert_y.shape[0], orig_y.shape[0]):
        raise IndexError(f"sample_idx {sample_idx} out of range")

    Np, H, W, Tp = pert_y.shape
    No, Ho, Wo, To = orig_y.shape
    if (H, W) != (Ho, Wo):
        raise ValueError(f"spatial size mismatch: pert {(H,W)} vs orig {(Ho,Wo)}")

    T = min(Tp, To)
    max_frames_per_row = rows * cols
    num_frames = min(T, max_frames_per_row)

    # 取出数据为 numpy
    p = pert_y[sample_idx].detach().cpu().numpy()   # (H, W, Tp)
    o = orig_y[sample_idx].detach().cpu().numpy()   # (H, W, To)
    diff = p[..., :T] - o[..., :T]

    total_cols = cols * 3
    fig, axes = plt.subplots(rows, total_cols,
                             figsize=(total_cols * 1.5, rows * 1.5),
                             constrained_layout=True)
    axes = np.asarray(axes)

    # 如果 shared_scale=True，则算一个共同 vmin/vmax
    if shared_scale:
        vmin = min(p.min(), o.min(), diff.min())
        vmax = max(p.max(), o.max(), diff.max())
        # 共享的限制
        vlim_p = vlim_o = vlim_d = (vmin, vmax)
    # else: 我们不预先计算共享的 vlim；按每张图自己取自己的 vmin/vmax

    for r in range(rows):
        for c in range(cols):
            frame_idx = r * cols + c
            if frame_idx >= num_frames:
                # 隐藏
                for k in range(3):
                    axes[r, 3*c + k].set_visible(False)
                continue

            # pert 图
            ax_p = axes[r, 3*c + 0]
            img_p = p[..., frame_idx]
            if shared_scale:
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=vlim_p[0], vmax=vlim_p[1])
            else:
                v0, v1 = img_p.min(), img_p.max()
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=v0, vmax=v1)
            ax_p.set_title(f"pert t={frame_idx}", fontsize=9)
            ax_p.axis("off")

            # orig 图
            ax_o = axes[r, 3*c + 1]
            img_o = o[..., frame_idx]
            if shared_scale:
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=vlim_o[0], vmax=vlim_o[1])
            else:
                v0, v1 = img_o.min(), img_o.max()
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=v0, vmax=v1)
            ax_o.set_title(f"orig t={frame_idx}", fontsize=9)
            ax_o.axis("off")

            # diff 图
            ax_d = axes[r, 3*c + 2]
            img_d = diff[..., frame_idx]
            if shared_scale:
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=vlim_d[0], vmax=vlim_d[1])
            else:
                v0, v1 = img_d.min(), img_d.max()
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=v0, vmax=v1)
            ax_d.set_title(f"diff t={frame_idx}", fontsize=9)
            ax_d.axis("off")

    fig.suptitle(f"{data_name} sample index={sample_idx}", fontsize=12)
    return fig


def plot_pert_orig_diff_side_by_side_blocks(
    pert_y: torch.Tensor,
    orig_y: torch.Tensor,
    sample_idx: int = 0,
    rows: int = 5,
    cols: int = 5,                 # 每块宽度（帧数）为 cols
    cmap_pert: str = "viridis",
    cmap_orig: str = "viridis",
    cmap_diff: str = "coolwarm",
    origin: str = "lower",
    data_name: str = "",
    shared_scale: bool = True
) -> plt.Figure:
    """
    块布局版本：按三块（pert / orig / diff）排列，每块宽度 cols。
    shared_scale=True 时三者共用 color scale，
    shared_scale=False 时每张子图自己取色标。
    """
    if pert_y.ndim != 4 or orig_y.ndim != 4:
        raise ValueError(f"expect (N,H,W,T), got {tuple(pert_y.shape)} and {tuple(orig_y.shape)}")
    if sample_idx < 0 or sample_idx >= min(pert_y.shape[0], orig_y.shape[0]):
        raise IndexError(f"sample_idx {sample_idx} out of range")

    Np, H, W, Tp = pert_y.shape
    No, Ho, Wo, To = orig_y.shape
    if (H, W) != (Ho, Wo):
        raise ValueError(f"spatial size mismatch: pert {(H,W)} vs orig {(Ho,Wo)}")

    T = min(Tp, To)
    max_frames = rows * cols
    num_frames = min(T, max_frames)

    p = pert_y[sample_idx].detach().cpu().numpy()
    o = orig_y[sample_idx].detach().cpu().numpy()
    diff = p[..., :T] - o[..., :T]

    total_cols = cols * 3
    fig, axes = plt.subplots(rows, total_cols,
                             figsize=(total_cols * 1.5, rows * 1.5),
                             constrained_layout=True)
    axes = np.asarray(axes)

    if shared_scale:
        vmin = min(p.min(), o.min(), diff.min())
        vmax = max(p.max(), o.max(), diff.max())
        vlim_p = vlim_o = vlim_d = (vmin, vmax)

    for r in range(rows):
        for c in range(cols):
            frame_idx = r * cols + c
            if frame_idx >= num_frames:
                axes[r, c].set_visible(False)
                axes[r, c + cols].set_visible(False)
                axes[r, c + 2*cols].set_visible(False)
                continue

            # pert 块
            ax_p = axes[r, c]
            img_p = p[..., frame_idx]
            if shared_scale:
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=vlim_p[0], vmax=vlim_p[1])
            else:
                v0, v1 = img_p.min(), img_p.max()
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=v0, vmax=v1)
            ax_p.set_title(f"pert t={frame_idx}", fontsize=9)
            ax_p.axis("off")

            # orig 块
            ax_o = axes[r, c + cols]
            img_o = o[..., frame_idx]
            if shared_scale:
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=vlim_o[0], vmax=vlim_o[1])
            else:
                v0, v1 = img_o.min(), img_o.max()
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=v0, vmax=v1)
            ax_o.set_title(f"orig t={frame_idx}", fontsize=9)
            ax_o.axis("off")

            # diff 块
            ax_d = axes[r, c + 2*cols]
            img_d = diff[..., frame_idx]
            if shared_scale:
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=vlim_d[0], vmax=vlim_d[1])
            else:
                v0, v1 = img_d.min(), img_d.max()
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=v0, vmax=v1)
            ax_d.set_title(f"diff t={frame_idx}", fontsize=9)
            ax_d.axis("off")

    fig.suptitle(f"{data_name} sample index={sample_idx}", fontsize=12)
    return fig


In [ ]:
import os
import torch

# 假设 plot_pert_orig_diff_side_by_side 和 plot_pert_orig_diff_side_by_side_blocks 这两个函数
# 已按你需要的布局写好，并返回 matplotlib.figure.Figure

root_out = "figures"
os.makedirs(root_out, exist_ok=True)

# for data_name in ["diagonal","ringsCos","none","isoCircles","petals","ringsL1","sBands"]:
for data_name in ["diagonalinit0"]:
    perturbed_data = torch.load(f"expanded_datasets/{data_name}_attack_N1150_train.pt", weights_only=False)
    original_data = torch.load(f"all_patterns/{data_name}/dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPattern{data_name}.pt", weights_only=False)

    # 为每个 data_name 创建子目录
    out_dir = os.path.join(root_out, data_name)
    os.makedirs(out_dir, exist_ok=True)

    for i in [0,1,2,3,4]:

        # 2. shared_scale = False
        fig = plot_pert_orig_diff_side_by_side(
            pert_y = perturbed_data["y"],
            orig_y = original_data["y"],
            sample_idx = i,
            rows = 5, cols = 5,
            cmap_pert = "viridis", cmap_orig = "viridis", cmap_diff = "coolwarm",
            origin = "lower",
            data_name = data_name,
            shared_scale = False
        )
        fname = f"{data_name}_idx{i}_noshared.png"
        path = os.path.join(out_dir, fname)
        fig.savefig(path)
        plt.close(fig)

        # 4. 用 block 布局，shared_scale = False
        fig = plot_pert_orig_diff_side_by_side_blocks(
            pert_y = perturbed_data["y"],
            orig_y = original_data["y"],
            sample_idx = i,
            rows = 5, cols = 5,
            cmap_pert = "viridis", cmap_orig = "viridis", cmap_diff = "coolwarm",
            origin = "lower",
            data_name = data_name,
            shared_scale = False
        )
        fname = f"{data_name}_idx{i}_block_noshared.png"
        path = os.path.join(out_dir, fname)
        fig.savefig(path)
        plt.close(fig)


In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from typing import Optional

def plot_pert_orig_diff_side_by_side_avg(
    pert_y: torch.Tensor,         # shape: (N, H, W, T)
    orig_y: torch.Tensor,         # shape: (N, H, W, T)
    rows: int = 5,
    cols: int = 5,                # 每行 cols 帧
    cmap_pert: str = "viridis",
    cmap_orig: str = "viridis",
    cmap_diff: str = "coolwarm",
    origin: str = "lower",
    data_name: str = "",
    shared_scale: bool = True     # 是否共享 color scale
) -> plt.Figure:
    """
    平均版本 + 并排 (pert / orig / diff) 布局。
    shared_scale=True => 三图共用 vmin/vmax；
    shared_scale=False => 每张子图各自色标。
    """
    # 检查维度
    if pert_y.ndim != 4 or orig_y.ndim != 4:
        raise ValueError(f"expect (N,H,W,T), got {tuple(pert_y.shape)} and {tuple(orig_y.shape)}")
    Np, H, W, Tp = pert_y.shape
    No, Ho, Wo, To = orig_y.shape
    if (H, W) != (Ho, Wo):
        raise ValueError(f"spatial size mismatch: pert {(H,W)} vs orig {(Ho,Wo)}")

    # 取最小 T
    T = min(Tp, To)
    max_frames = rows * cols
    num_frames = min(T, max_frames)

    # 平均 over N
    p = pert_y.mean(dim=0).detach().cpu().numpy()  # (H, W, T)
    o = orig_y.mean(dim=0).detach().cpu().numpy()
    diff = p[..., :T] - o[..., :T]

    total_cols = cols * 3
    fig, axes = plt.subplots(rows, total_cols,
                             figsize=(total_cols * 1.5, rows * 1.5),
                             constrained_layout=True)
    axes = np.asarray(axes)

    # 如果共享色标，则预先计算一个 global vmin / vmax
    if shared_scale:
        vmin = min(p.min(), o.min(), diff.min())
        vmax = max(p.max(), o.max(), diff.max())
        vlim_p = vlim_o = vlim_d = (vmin, vmax)

    for r in range(rows):
        for c in range(cols):
            frame_idx = r * cols + c
            if frame_idx >= num_frames:
                axes[r, 3*c + 0].set_visible(False)
                axes[r, 3*c + 1].set_visible(False)
                axes[r, 3*c + 2].set_visible(False)
                continue

            # pert 图
            ax_p = axes[r, 3*c + 0]
            img_p = p[..., frame_idx]
            if shared_scale:
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=vlim_p[0], vmax=vlim_p[1])
            else:
                v0, v1 = img_p.min(), img_p.max()
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=v0, vmax=v1)
            ax_p.set_title(f"pert t={frame_idx}", fontsize=9)
            ax_p.axis("off")

            # orig 图
            ax_o = axes[r, 3*c + 1]
            img_o = o[..., frame_idx]
            if shared_scale:
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=vlim_o[0], vmax=vlim_o[1])
            else:
                v0, v1 = img_o.min(), img_o.max()
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=v0, vmax=v1)
            ax_o.set_title(f"orig t={frame_idx}", fontsize=9)
            ax_o.axis("off")

            # diff 图
            ax_d = axes[r, 3*c + 2]
            img_d = diff[..., frame_idx]
            if shared_scale:
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=vlim_d[0], vmax=vlim_d[1])
            else:
                v0, v1 = img_d.min(), img_d.max()
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=v0, vmax=v1)
            ax_d.set_title(f"diff t={frame_idx}", fontsize=9)
            ax_d.axis("off")

    fig.suptitle(f"{data_name} avg (shared={shared_scale})")
    return fig


def plot_pert_orig_diff_blocks_avg(
    pert_y: torch.Tensor,
    orig_y: torch.Tensor,
    rows: int = 5,
    cols: int = 5,
    cmap_pert: str = "viridis",
    cmap_orig: str = "viridis",
    cmap_diff: str = "coolwarm",
    origin: str = "lower",
    data_name: str = "",
    shared_scale: bool = True
) -> plt.Figure:
    """
    平均版本 + 块布局（pert / orig / diff 三块排列）。
    shared_scale=True => 三块共用色标；
    shared_scale=False => 每个子图独立色标。
    """
    if pert_y.ndim != 4 or orig_y.ndim != 4:
        raise ValueError(f"expect (N,H,W,T), got {tuple(pert_y.shape)} and {tuple(orig_y.shape)}")
    Np, H, W, Tp = pert_y.shape
    No, Ho, Wo, To = orig_y.shape
    if (H, W) != (Ho, Wo):
        raise ValueError(f"spatial size mismatch: pert {(H,W)} vs orig {(Ho,Wo)}")

    T = min(Tp, To)
    max_frames = rows * cols
    num_frames = min(T, max_frames)

    # 平均 over N
    p = pert_y.mean(dim=0).detach().cpu().numpy()
    o = orig_y.mean(dim=0).detach().cpu().numpy()
    diff = p[..., :T] - o[..., :T]

    total_cols = cols * 3
    fig, axes = plt.subplots(rows, total_cols,
                             figsize=(total_cols * 1.5, rows * 1.5),
                             constrained_layout=True)
    axes = np.asarray(axes)

    if shared_scale:
        vmin = min(p.min(), o.min(), diff.min())
        vmax = max(p.max(), o.max(), diff.max())
        vlim_p = vlim_o = vlim_d = (vmin, vmax)

    for r in range(rows):
        for c in range(cols):
            frame_idx = r * cols + c
            if frame_idx >= num_frames:
                # 隐藏多余子图
                axes[r, c].set_visible(False)
                axes[r, c + cols].set_visible(False)
                axes[r, c + 2*cols].set_visible(False)
                continue

            # pert 块
            ax_p = axes[r, c]
            img_p = p[..., frame_idx]
            if shared_scale:
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=vlim_p[0], vmax=vlim_p[1])
            else:
                v0, v1 = img_p.min(), img_p.max()
                ax_p.imshow(img_p, origin=origin, cmap=cmap_pert,
                            vmin=v0, vmax=v1)
            ax_p.set_title(f"pert t={frame_idx}", fontsize=9)
            ax_p.axis("off")

            # orig 块
            ax_o = axes[r, c + cols]
            img_o = o[..., frame_idx]
            if shared_scale:
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=vlim_o[0], vmax=vlim_o[1])
            else:
                v0, v1 = img_o.min(), img_o.max()
                ax_o.imshow(img_o, origin=origin, cmap=cmap_orig,
                            vmin=v0, vmax=v1)
            ax_o.set_title(f"orig t={frame_idx}", fontsize=9)
            ax_o.axis("off")

            # diff 块
            ax_d = axes[r, c + 2*cols]
            img_d = diff[..., frame_idx]
            if shared_scale:
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=vlim_d[0], vmax=vlim_d[1])
            else:
                v0, v1 = img_d.min(), img_d.max()
                ax_d.imshow(img_d, origin=origin, cmap=cmap_diff,
                            vmin=v0, vmax=v1)
            ax_d.set_title(f"diff t={frame_idx}", fontsize=9)
            ax_d.axis("off")

    fig.suptitle(f"{data_name} avg (shared={shared_scale})")
    return fig


In [ ]:
import os

root_out = "figures_avg"
os.makedirs(root_out, exist_ok=True)

# for data_name in ["diagonal","ringsCos","none","isoCircles","petals","ringsL1","sBands"]:
for data_name in ["diagonalinit0"]:
    perturbed = torch.load(f"expanded_datasets/{data_name}_attack_N1150_train.pt", weights_only=False)
    original = torch.load(f"all_patterns/{data_name}/dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPattern{data_name}.pt", weights_only=False)

    out_sub = os.path.join(root_out, data_name)
    os.makedirs(out_sub, exist_ok=True)

    for shared in [False]:
        # 三合并并排版本
        fig = plot_pert_orig_diff_side_by_side_avg(
            pert_y=perturbed["y"],
            orig_y=original["y"],
            rows=5, cols=5,
            cmap_pert="viridis",
            cmap_orig="viridis",
            cmap_diff="coolwarm",
            origin="lower",
            data_name=data_name,
            shared_scale=shared
        )
        fname = f"{data_name}_avg_shared{shared}.png"
        fig.savefig(os.path.join(out_sub, fname), dpi=300)
        plt.close(fig)

        # 块布局版本
        fig2 = plot_pert_orig_diff_blocks_avg(
            pert_y=perturbed["y"],
            orig_y=original["y"],
            rows=5, cols=5,
            cmap_pert="viridis",
            cmap_orig="viridis",
            cmap_diff="coolwarm",
            origin="lower",
            data_name=data_name,
            shared_scale=shared
        )
        fname2 = f"{data_name}_avg_block_shared{shared}.png"
        fig2.savefig(os.path.join(out_sub, fname2), dpi=300)
        plt.close(fig2)
